# v80 -- GZ2 Spiral Winding Asymmetry Along KTF³ Axis
## KTF3-R axis l=277, b=-31: tight vs loose arms along KTF³ axis
**Author:** Andrew Cotting | 5 April 2026 | github.com/Andrew-Cot/KTF3-Analysis

### Context
gz2_hart16.csv (Hart et al. 2016) contains debiased spiral arm winding fractions.
KTF³-R predicts a preferred direction l=277, b=-31 (galactic frame).
Test: Is spiral arm winding (tight vs loose) asymmetric along KTF³ axis?

### KTF³ Prediction
The screw isometry φ_R rotates θ_R = 25.7° per cycle and translates T₁ = 1660 Mpc.
Spirals in the KTF³ northern hemisphere should show systematic winding difference
vs southern hemisphere (topological chirality imprint).

### Data
gz2_hart16.csv — Hart et al. 2016, debiased GZ2 morphology catalog
Key columns: t10_arms_winding_a28_tight_debiased, t10_arms_winding_a30_loose_debiased
Requires: gz2_hart16.csv uploaded to Colab

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os

print('='*65)
print('v80 -- GZ2 Spiral Winding Asymmetry Along KTF³ Axis')
print('KTF³ axis: l=277 deg, b=-31 deg (galactic frame)')
print('Data: gz2_hart16.csv (Hart et al. 2016)')
print('='*65)

# KTF³ axis
l_ktf3, b_ktf3 = 277.0, -31.0

def gal_to_vec(l, b):
    l_r, b_r = np.radians(l), np.radians(b)
    return np.array([np.cos(b_r)*np.cos(l_r),
                     np.cos(b_r)*np.sin(l_r),
                     np.sin(b_r)])

def radec_to_gal(ra, dec):
    ra_r, dec_r = np.radians(ra), np.radians(dec)
    ra_NGP = np.radians(192.85948)
    dec_NGP = np.radians(27.12825)
    l_NCP = 122.93192
    sin_b = (np.sin(dec_r)*np.sin(dec_NGP) +
             np.cos(dec_r)*np.cos(dec_NGP)*np.cos(ra_r - ra_NGP))
    b = np.degrees(np.arcsin(np.clip(sin_b, -1, 1)))
    x = np.cos(dec_r)*np.sin(ra_r - ra_NGP)
    y = (np.sin(dec_r)*np.cos(dec_NGP) -
         np.cos(dec_r)*np.sin(dec_NGP)*np.cos(ra_r - ra_NGP))
    l = (l_NCP - np.degrees(np.arctan2(x, y))) % 360
    return l, b

axis_vec = gal_to_vec(l_ktf3, b_ktf3)
print(f'KTF³ axis vector: {axis_vec.round(4)}')


In [ ]:
# Load gz2_hart16.csv
gz2_files = ['gz2_hart16.csv', 'gz2_hart16_debiased_vote_fractions.csv',
             'gz2_hart16_debiased_vote_fractions.fits']
gz2_file = next((f for f in gz2_files if os.path.exists(f)), None)

if gz2_file is None:
    raise FileNotFoundError(
        'gz2_hart16.csv not found!\n'
        'Please upload gz2_hart16.csv to Colab first.\n'
        'Download from: https://data.galaxyzoo.org/'
    )

print(f'Loading: {gz2_file} ...')
df = pd.read_csv(gz2_file)
print(f'Loaded {len(df):,} galaxies, {len(df.columns)} columns')

# Required columns
required = [
    'ra', 'dec',
    't04_spiral_a08_spiral_debiased',
    't10_arms_winding_a28_tight_debiased',
    't10_arms_winding_a29_medium_debiased',
    't10_arms_winding_a30_loose_debiased',
]
for col in required:
    if col not in df.columns:
        raise ValueError(f'Missing column: {col}')

print('All required columns present ✓')
print(f'Sample row:\n{df[required].iloc[0]}')


In [ ]:
# Select spiral galaxies with reliable winding data
# Criteria: spiral debiased > 0.5, tight+loose > 0.3
mask_spiral = df['t04_spiral_a08_spiral_debiased'] > 0.5
mask_winding = (df['t10_arms_winding_a28_tight_debiased'] +
                df['t10_arms_winding_a30_loose_debiased']) > 0.3
df_sp = df[mask_spiral & mask_winding].copy()
print(f'Spiral galaxies with winding data: {len(df_sp):,}')

# Convert RA/Dec to galactic
print('Converting to galactic coordinates...')
l_gal, b_gal = radec_to_gal(df_sp['ra'].values, df_sp['dec'].values)
df_sp['l_gal'] = l_gal
df_sp['b_gal'] = b_gal

# Galactic plane mask: |b| > 10 deg
mask_gal = np.abs(df_sp['b_gal']) > 10
df_sp = df_sp[mask_gal].copy()
print(f'After |b|>10 deg mask: {len(df_sp):,}')

# Unit vectors for each galaxy
l_r = np.radians(df_sp['l_gal'].values)
b_r = np.radians(df_sp['b_gal'].values)
vecs = np.column_stack([
    np.cos(b_r)*np.cos(l_r),
    np.cos(b_r)*np.sin(l_r),
    np.sin(b_r)
])

# Dot product with KTF³ axis
dot_axis = vecs @ axis_vec
df_sp['dot_ktf3'] = dot_axis

# Winding index: tight=+1, loose=-1 (signed winding measure)
tight = df_sp['t10_arms_winding_a28_tight_debiased'].values
loose = df_sp['t10_arms_winding_a30_loose_debiased'].values
winding_index = tight - loose  # positive = tight, negative = loose
df_sp['winding_index'] = winding_index

print(f'\nWinding index statistics:')
print(f'  Mean: {winding_index.mean():.4f}')
print(f'  Std:  {winding_index.std():.4f}')
print(f'  Tight dominant: {(winding_index>0).sum():,}')
print(f'  Loose dominant: {(winding_index<0).sum():,}')


In [ ]:
# HEMISPHERIC TEST: KTF³ north vs south
north = dot_axis > 0
south = dot_axis < 0

wi_north = winding_index[north]
wi_south = winding_index[south]

mean_north = wi_north.mean()
mean_south = wi_south.mean()
diff = mean_north - mean_south

# Significance via bootstrap
print('Bootstrap significance test (1000 iterations)...')
np.random.seed(277)
n_boot = 1000
boot_diffs = []
n_n, n_s = north.sum(), south.sum()
for _ in range(n_boot):
    perm = np.random.permutation(winding_index)
    boot_diffs.append(perm[:n_n].mean() - perm[n_n:n_n+n_s].mean())

boot_diffs = np.array(boot_diffs)
sigma = (diff - boot_diffs.mean()) / boot_diffs.std()

print(f'\n{"="*55}')
print(f'HEMISPHERIC WINDING ASYMMETRY TEST (REAL GZ2 DATA)')
print(f'{"="*55}')
print(f'N galaxies total:  {len(winding_index):,}')
print(f'N north (dot>0):   {n_n:,}')
print(f'N south (dot<0):   {n_s:,}')
print(f'Mean winding north: {mean_north:+.5f} (positive=tight)')
print(f'Mean winding south: {mean_south:+.5f}')
print(f'Difference (N-S):   {diff:+.5f}')
print(f'Bootstrap mean:     {boot_diffs.mean():+.5f}')
print(f'Bootstrap std:      {boot_diffs.std():.5f}')
print(f'SIGMA:              {sigma:+.3f}')
verdict = 'SIGNAL' if abs(sigma) >= 2.0 else ('HINT' if abs(sigma) >= 1.5 else 'NULL')
print(f'VERDICT:            {verdict}')
print(f'{"="*55}')


In [ ]:
# ANGULAR PROFILE: winding vs angle from KTF³ axis
angle_deg = np.degrees(np.arccos(np.clip(dot_axis, -1, 1)))
bins = np.linspace(0, 180, 13)  # 15-deg bins
centers = 0.5*(bins[1:]+bins[:-1])

mean_w, err_w, n_bin = [], [], []
for i in range(len(bins)-1):
    m = (angle_deg >= bins[i]) & (angle_deg < bins[i+1])
    if m.sum() > 50:
        mean_w.append(winding_index[m].mean())
        err_w.append(winding_index[m].std() / np.sqrt(m.sum()))
        n_bin.append(m.sum())
    else:
        mean_w.append(np.nan)
        err_w.append(np.nan)
        n_bin.append(0)

mean_w = np.array(mean_w)
err_w = np.array(err_w)

# Pearson correlation: winding index vs dot product
corr = np.corrcoef(dot_axis, winding_index)[0,1]
print(f'Pearson correlation (dot_axis, winding_index): r = {corr:.4f}')

# Plot
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.suptitle(
    f'v80 -- GZ2 Spiral Winding Asymmetry Along KTF³ Axis\n'
    f'KTF³: l=277°, b=-31° | N={len(winding_index):,} spirals | '
    f'σ={sigma:+.2f} | REAL DATA (Hart et al. 2016)',
    fontsize=12, fontweight='bold'
)

ax1 = axes[0]
ax1.errorbar(centers, mean_w, yerr=err_w, fmt='b-o', lw=2, ms=7, capsize=4)
ax1.axhline(0, color='black', lw=1)
ax1.axvline(90, color='red', linestyle='--', lw=1.5, label='KTF³ equator')
ax1.axhline(winding_index.mean(), color='gray', linestyle=':', label='Global mean')
ax1.set_xlabel('Angular distance from KTF³ axis [deg]', fontsize=11)
ax1.set_ylabel('Mean winding index (tight - loose)', fontsize=11)
ax1.set_title('Winding vs angle from KTF³ axis', fontsize=11)
ax1.legend(); ax1.grid(True, alpha=0.3)

ax2 = axes[1]
ax2.scatter(dot_axis[::10], winding_index[::10], alpha=0.1, s=1, c='steelblue')
z = np.polyfit(dot_axis, winding_index, 1)
xfit = np.linspace(-1, 1, 100)
ax2.plot(xfit, np.polyval(z, xfit), 'r-', lw=2, label=f'r={corr:.4f}')
ax2.axhline(0, color='black', lw=0.5)
ax2.axvline(0, color='red', linestyle='--', lw=1, label='KTF³ equator')
ax2.set_xlabel('cos(angle) with KTF³ axis', fontsize=11)
ax2.set_ylabel('Winding index (tight - loose)', fontsize=11)
ax2.set_title(f'Scatter: winding vs KTF³ alignment\nr={corr:.4f}', fontsize=11)
ax2.legend(); ax2.grid(True, alpha=0.3)

ax3 = axes[2]
ax3.hist(boot_diffs, bins=40, color='#95a5a6', alpha=0.8, edgecolor='black',
         density=True, label='Bootstrap (null)')
ax3.axvline(diff, color='green', lw=2.5, label=f'Observed Δ={diff:+.5f}')
ax3.axvline(boot_diffs.mean()+boot_diffs.std(), color='orange', linestyle='--', label='1σ')
ax3.axvline(boot_diffs.mean()-boot_diffs.std(), color='orange', linestyle='--')
ax3.axvline(boot_diffs.mean()+2*boot_diffs.std(), color='red', linestyle=':', label='2σ')
ax3.axvline(boot_diffs.mean()-2*boot_diffs.std(), color='red', linestyle=':')
ax3.set_xlabel('N-S winding difference', fontsize=11)
ax3.set_ylabel('Density', fontsize=11)
ax3.set_title(f'Hemispheric bootstrap test\nσ = {sigma:+.3f}  [{verdict}]', fontsize=11)
ax3.legend(fontsize=9); ax3.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('v80_gz2_refined.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nGitHub: github.com/Andrew-Cot/KTF3-Analysis | OSF: osf.io/42nks')
print('Data: Hart et al. 2016, GZ2 debiased morphology catalog')
